# Vertex AI Vector Search Lab
Complete, tested end-to-end pipeline for semantic book search.

In [ ]:
# Step 1: Install required Google Cloud SDK packages (run once)
%pip install -q --upgrade google-cloud-aiplatform google-cloud-storage

In [ ]:
# Step 2: Initialize Vertex AI & Configuration
import uuid
from google.cloud import aiplatform

PROJECT_ID = "jovial-coral-423517-h8"
LOCATION = "us-central1"
BUCKET_NAME = "tuc-vector-search-bucket-1"
BLOB_NAME = "books.txt"

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print("Vertex AI initialized successfully.")

In [ ]:
# Step 3: Create GCS Bucket, Upload books.txt, and Load into Memory
import os
from google.cloud import storage

LOCAL_FILE_PATH = r"C:\Users\hmira\GitHub-Org\teachucode-courses\Google-Gemini-ADK-Agents-Lab\Lab18-vector-search\books.txt"

storage_client = storage.Client(project=PROJECT_ID)

# 1. Create bucket if it doesn't exist
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(bucket, location=LOCATION)
    print(f"Bucket '{BUCKET_NAME}' created in {LOCATION}.")
else:
    print(f"Bucket '{BUCKET_NAME}' ready.")

# 2. Upload books.txt if not already present
blob = bucket.blob(BLOB_NAME)
if not blob.exists() and os.path.exists(LOCAL_FILE_PATH):
    blob.upload_from_filename(LOCAL_FILE_PATH)
    print(f"Uploaded '{LOCAL_FILE_PATH}' to GCS.")

# 3. Download text and clean lines
text_data = blob.download_as_text()
raw_lines = [line.strip() for line in text_data.split("\n") if line.strip()]
documents = [doc for doc in raw_lines if doc != "Book-Title"]

print(f"Loaded {len(documents)} books ready for embedding!")

In [ ]:
# Step 4: Generate Embeddings in Batches (Prevents 429 Quota Exceeded)
import time
from vertexai.language_models import TextEmbeddingModel

embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

batch_size = 100
embeddings = []

print(f"Generating embeddings for {len(documents)} books in batches of {batch_size}...")
for i in range(0, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    print(f"Embedding batch {i // batch_size + 1}: books {i + 1} to {min(i + batch_size, len(documents))}...")
    response = embedding_model.get_embeddings(batch)
    embeddings.extend([emb.values for emb in response])
    if i + batch_size < len(documents):
        time.sleep(1.0)

print(f"\nSuccess! Created {len(embeddings)} embeddings of dimension {len(embeddings[0])}.")

In [ ]:
# Step 5: Create Vector Search Index (Tree-AH with Leaf Node Config)
# Note: Takes ~15-25 minutes to complete
index_id = f"my-book-index-{uuid.uuid4().hex[:6]}"
print(f"Creating index: {index_id}")

index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=index_id,
    dimensions=len(embeddings[0]),          # 768
    approximate_neighbors_count=10,         # Required for Tree-AH
    distance_measure_type="COSINE_DISTANCE",
    leaf_node_embedding_count=500,          # Fixes 400 algorithmConfig error
    leaf_nodes_to_search_percent=10,        # Fixes 400 algorithmConfig error
    shard_size="SHARD_SIZE_SMALL",          # Cost optimization
    index_update_method="STREAM_UPDATE",    # Allows real-time upserts
    sync=True
)

print("Waiting for index creation to complete...")
index.wait()
print("Index created successfully!")

In [ ]:
# Step 6: Create Endpoint and Deploy Index
# Note: Takes ~10-15 minutes to complete
endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"{index_id}-endpoint",
    public_endpoint_enabled=True,
)

print("Deploying index to endpoint (please wait)...\n")
endpoint.deploy_index(
    index=index,
    deployed_index_id="my_unique_deployment_id_v2"
)
print("Index successfully deployed to endpoint!")

In [ ]:
# Step 7: Upsert Book Datapoints into the Live Index
from google.cloud.aiplatform_v1.types import index as index_types

datapoints = [
    index_types.IndexDatapoint(
        datapoint_id=str(i),
        feature_vector=emb,
    )
    for i, emb in enumerate(embeddings)
]

print(f"Upserting {len(datapoints)} vectors into the index...")
index.upsert_datapoints(datapoints=datapoints)
print(f"Successfully upserted {len(datapoints)} vectors!")

In [ ]:
# Step 8: Perform Semantic Search
query_text = "find similar to Bible"
query_emb = embedding_model.get_embeddings([query_text])[0].values

# Search the 5 nearest neighbors
neighbors_list = endpoint.find_neighbors(
    deployed_index_id="my_unique_deployment_id_v2",
    queries=[query_emb],
    num_neighbors=5,
)

print(f"Query: '{query_text}'\nTop Matches:")
for neighbor in neighbors_list[0]:
    idx = int(neighbor.id)
    similarity = (1 - neighbor.distance) * 100
    print(f"- {documents[idx]} (Distance: {neighbor.distance:.4f} | Match: {similarity:.1f}%)")

In [ ]:
# Step 9: (Optional) Inspect Raw Vectors Stored in the Live Endpoint
stored_points = endpoint.read_index_datapoints(
    deployed_index_id="my_unique_deployment_id_v2",
    ids=["0", "1"]
)

for pt in stored_points:
    print(f"ID: {pt.datapoint_id} | Vector Dimension: {len(pt.feature_vector)} | First 5 values: {list(pt.feature_vector)[:5]}")

In [ ]:
# Step 10: Complete Cleanup (Stops All Ongoing Charges)
# IMPORTANT: Run this when you are done to avoid ~$15-$20/day endpoint billing!
print("Undeploying and deleting endpoint...")
endpoint.delete(force=True)
print("Endpoint deleted.")

print("Deleting index...")
index.delete()
print("Index deleted. All billing stopped!")